# Step 1 — Setup & Data Loading

In [53]:
!pip install scikit-learn pandas numpy --quiet

import json
import numpy as np
import pandas as pd
from google.colab import files

print("✅ Libraries loaded successfully")

uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"✅ Loaded {len(raw_data)} places from '{filename}'")

df = pd.DataFrame(raw_data)

✅ Libraries loaded successfully


Saving places_all_v1.json to places_all_v1 (2).json
✅ Loaded 266 places from 'places_all_v1 (2).json'


In [54]:
df.shape

(266, 25)

In [55]:
list(df.columns)

['id',
 'name_ar',
 'name_en',
 'category',
 'location',
 'full_address',
 'opening_hours',
 'description',
 'best_time_to_visit',
 'recommended_duration',
 'image_url',
 'accessibility',
 'indoor_or_outdoor',
 'price_egp',
 'price_usd',
 'price_category',
 'rating',
 'number_of_reviews',
 'popularity_score',
 'latitude',
 'longitude',
 'tags',
 'suitable_for',
 'nearby_attractions',
 'search_text']

In [56]:
print(f"\n Governorates found:")
print(df["location"].value_counts().head(20))


 Governorates found:
location
Cairo / القاهرة                             56
Haram / Haram                               53
Luxor / الأقصر                              33
West Bank — Luxor / البر الغربي — الأقصر    29
Islamic Cairo / القاهرة الإسلامية           27
Dokki / Dokki                               14
Giza / الجيزة                               10
Fayoum / Fayoum                              8
Giza / Giza                                  7
Coptic Cairo / القاهرة القبطية               6
6th October / 6th October                    5
Saqqara / Saqqara                            4
Sheikh Zayed / Sheikh Zayed                  3
Manial / Manial                              2
Agouza / Agouza                              2
Heliopolis / مصر الجديدة                     1
Dahshur / Dahshur                            1
badrashin / Badrashin                        1
North Luxor / شمال الأقصر                    1
Sohag — Near Luxor / سوهاج — قرب الأقصر      1
Name: count, dtype: int64


In [57]:
print(f"\n Price range (EGP): {df['price_egp'].min()} → {df['price_egp'].max()}")
print(f" Rating range: {df['rating'].min()} → {df['rating'].max()}")


 Price range (EGP): 0.0 → 8000.0
 Rating range: 3.7 → 4.9


In [58]:
print(f"\n Null values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])


 Null values per column:
Series([], dtype: int64)


# Step 2 — Preprocessing & Feature Engineering


In [59]:
import re

# --- 2.1 Governorate Mapping ---
GOVERNORATE_MAP = {
    # Cairo
    "cairo"         : "Cairo",
    "islamic cairo" : "Cairo",
    "coptic cairo"  : "Cairo",
    "heliopolis"    : "Cairo",
    "manial"        : "Cairo",
    "agouza"        : "Cairo",
    "dokki"         : "Cairo",

    # Giza
    "giza"          : "Giza",
    "haram"         : "Giza",
    "6th october"   : "Giza",
    "sheikh zayed"  : "Giza",
    "saqqara"       : "Giza",
    "dahshur"       : "Giza",
    "badrashin"     : "Giza",
    "fayoum"        : "Giza",

    # Luxor
    "luxor"         : "Luxor",
    "west bank"     : "Luxor",
    "north luxor"   : "Luxor",
    "sohag"         : "Luxor",

    # Aswan
    "aswan"         : "Aswan",
}

def extract_governorate(location_str: str) -> str:
    loc_lower = location_str.lower()
    for keyword, gov in GOVERNORATE_MAP.items():
        if keyword in loc_lower:
            return gov
    return "Unknown"

df["governorate"] = df["location"].apply(extract_governorate)

print("Governorate distribution after mapping:")
print(df["governorate"].value_counts())
print(f"\n⚠️  Unknown locations: {(df['governorate'] == 'Unknown').sum()}")
if (df['governorate'] == 'Unknown').sum() > 0:
    print(df[df['governorate'] == 'Unknown'][['name_en', 'location']])

Governorate distribution after mapping:
governorate
Cairo    108
Giza      92
Luxor     66
Name: count, dtype: int64

⚠️  Unknown locations: 0


In [60]:
# --- 2.2 Parse recommended_duration → avg_duration_hours (float) ---
def parse_duration(duration_str: str) -> float:
    """
    '2–3 hours'  → 2.5
    '1 hour'     → 1.0
    '30 minutes' → 0.5
    """
    if not isinstance(duration_str, str):
        return 2.0  # default fallback

    duration_str = duration_str.lower().replace("–", "-").replace("—", "-")

    # range: "2-3 hours"
    range_match = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)", duration_str)
    if range_match:
        low, high = float(range_match.group(1)), float(range_match.group(2))
        hours = (low + high) / 2
        if "minute" in duration_str:
            hours /= 60
        return round(hours, 2)

    # single value: "2 hours" or "30 minutes"
    single_match = re.search(r"(\d+(?:\.\d+)?)", duration_str)
    if single_match:
        val = float(single_match.group(1))
        if "minute" in duration_str:
            return round(val / 60, 2)
        return round(val, 2)

    return 2.0  # fallback

df["avg_duration_hours"] = df["recommended_duration"].apply(parse_duration)

print("\n⏱️  Duration stats (hours):")
print(df["avg_duration_hours"].describe().round(2))
print("\nSample parsing check:")
sample = df[["name_en", "recommended_duration", "avg_duration_hours"]].sample(8, random_state=42)
print(sample.to_string(index=False))


⏱️  Duration stats (hours):
count    266.00
mean       1.92
std        0.88
min        0.50
25%        1.50
50%        2.00
75%        2.50
max        5.00
Name: avg_duration_hours, dtype: float64

Sample parsing check:
                        name_en recommended_duration  avg_duration_hours
 Arabian Horse Tour at Pyramids            2–3 hours                 2.5
Ancient Memphis Open Air Museum            1–2 hours                 1.5
        Pharaohs' Eco Farm Giza            2–3 hours                 2.5
    Felucca Nile Cruise — Luxor            1–2 hours                 1.5
                    Nile Museum            2–3 hours                 2.5
Upper Egyptian Pottery Workshop            1–2 hours                 1.5
       Wadi El-Rayan Waterfalls            3–4 hours                 3.5
          Tomb of Ti at Saqqara            1–2 hours                 1.5


In [61]:
# --- 2.3 Normalize tags → clean list of lowercase English keywords ---
def extract_tags(tags_val) -> list:
    """
    Input:  ["historical / تاريخي", "pharaonic / فرعوني"]
    Output: ["historical", "pharaonic"]
    """
    if not isinstance(tags_val, list):
        return []
    clean = []
    for tag in tags_val:
        # take only the English part (before '/')
        en_part = tag.split("/")[0].strip().lower()
        # keep alphanumeric + spaces only
        en_part = re.sub(r"[^a-z0-9 ]", "", en_part).strip()
        if en_part:
            clean.append(en_part)
    return clean

df["tags_clean"] = df["tags"].apply(extract_tags)

print("\n Tags sample:")
for _, row in df[["name_en", "tags_clean"]].sample(4, random_state=7).iterrows():
    print(f"  {row['name_en']}: {row['tags_clean']}")


 Tags sample:
  Royal Jewellery Museum: ['museum', 'royal', 'jewelry', 'culture', 'castle', 'deluxe', 'historical']
  Nubian Village Gharaba — Luxor: ['cultural', 'nubian', 'village', 'colorful', 'hospitality', 'traditional food', 'crafts', 'authentic']
  National Museum of Egyptian Civilization: ['museum', 'mummies', 'civilization', 'history', 'accident', 'culture', 'family']
  Medinet Habu — Ramesses III Temple: ['historical', 'pharaonic', 'temple', 'ramesses', 'colored reliefs', 'wellpreserved', 'quiet', 'rare']


In [62]:
# --- 2.4 Build master tag vocabulary ---
all_tags = set()
for tags in df["tags_clean"]:
    all_tags.update(tags)

print(f"\n Total unique tags in dataset: {len(all_tags)}")
print("Sample tags:", sorted(all_tags)[:20])

print("\n✅ Step 2 complete — DataFrame ready for Scoring Engine")


 Total unique tags in dataset: 389
Sample tags: ['abydos', 'accident', 'adventure', 'agriculture', 'alabaster', 'amenhotep', 'an experience', 'ancient', 'animals', 'annual', 'antique', 'antiques', 'aquarium', 'arabic', 'archaeological', 'archaeological sites', 'archaeology', 'architecture', 'arms', 'art']

✅ Step 2 complete — DataFrame ready for Scoring Engine


# Step 3 — Scoring Engine


In [63]:
# --- 3.1 User Input Definition ---
USER_INPUT = {
    "governorate" : "Luxor",           # Cairo | Giza | Luxor | Aswan
    "budget_egp"  : 3000,              # Total budget for tickets and activities
    "duration_days": 3,                # Number of trip days
    "interests"   : ["historical", "pharaonic", "temple", "museum"],
}

print(" User Input:")
for k, v in USER_INPUT.items():
    print(f"   {k}: {v}")

# --- 3.2 Step 1: Filter by Governorate ---
df_filtered = df[df["governorate"] == USER_INPUT["governorate"]].copy()

print(f"\n Places in {USER_INPUT['governorate']}: {len(df_filtered)}")

# --- 3.3 Step 2: Filter by Budget ---
daily_budget = USER_INPUT["budget_egp"] / USER_INPUT["duration_days"]

# Remove places whose ticket price alone exceeds 60% of the daily budget
budget_cap = daily_budget * 0.6

df_filtered = df_filtered[
    df_filtered["price_egp"] <= budget_cap
].copy()

print(f"\n Daily budget: {daily_budget:.0f} EGP | Price cap per place: {budget_cap:.0f} EGP")
print(f"   Places within budget: {len(df_filtered)}")

 User Input:
   governorate: Luxor
   budget_egp: 3000
   duration_days: 3
   interests: ['historical', 'pharaonic', 'temple', 'museum']

 Places in Luxor: 66

 Daily budget: 1000 EGP | Price cap per place: 600 EGP
   Places within budget: 60


In [64]:
# --- 3.4 Step 3: Compute Scores ---
def compute_score(row, interests: list, daily_budget: float) -> float:
    """
    Composite score based on 3 components:
      - Tag Match Score  (50%): How many user interests match the place tags
      - Budget Fit Score (25%): Cheaper places receive higher scores
      - Popularity Score (25%): Normalized popularity score from the dataset
    """

    # --- Tag Match (0.0 → 1.0) ---
    if not interests or not row["tags_clean"]:
        tag_score = 0.0
    else:
        matched = len(set(interests) & set(row["tags_clean"]))
        tag_score = matched / len(interests)

    # --- Budget Fit (0.0 → 1.0) ---
    # Free place = 1.0, place priced at budget_cap = 0.0
    if daily_budget > 0:
        budget_fit = 1.0 - (
            row["price_egp"] / (daily_budget * 0.6)
        )

        budget_fit = max(0.0, min(1.0, budget_fit))
    else:
        budget_fit = 1.0

    # --- Popularity (0.0 → 1.0) ---
    pop_score = row["popularity_score"] / 100.0

    # --- Weighted Final Score ---
    final_score = (
        0.50 * tag_score +
        0.25 * budget_fit +
        0.25 * pop_score
    )

    return round(final_score, 4)

df_filtered["score"] = df_filtered.apply(
    lambda row: compute_score(
        row,
        USER_INPUT["interests"],
        daily_budget
    ),
    axis=1
)

# --- 3.5 Sort & Select Top Candidates ---
# Select the best (duration_days × 4) places
# to give clustering enough options
max_places = USER_INPUT["duration_days"] * 4

df_scored = (
    df_filtered
    .sort_values("score", ascending=False)
    .head(max_places)
    .copy()
)

print(f"\n🏆 Top {max_places} scored places:")

display_cols = [
    "name_en",
    "governorate",
    "price_egp",
    "avg_duration_hours",
    "popularity_score",
    "score",
    "tags_clean"
]

print(df_scored[display_cols].to_string(index=False))

print(f"\n📊 Score stats:")
print(df_scored["score"].describe().round(3))


🏆 Top 12 scored places:
                                   name_en governorate  price_egp  avg_duration_hours  popularity_score  score                                                                                    tags_clean
                   Dendera Temple — Hathor       Luxor      200.0                2.50                85 0.7542 [historical, pharaonic, temple, dendera, hathor, astronomical ceiling, complete, magnificent]
                Mummification Museum Luxor       Luxor      150.0                1.50                75 0.7500                [museum, mummification, mummies, pharaonic, unique, science, historical, rare]
                 Temple of Seti I — Abydos       Luxor      200.0                2.50                83 0.7492        [historical, pharaonic, temple, seti, abydos, king list, colored reliefs, magnificent]
        Medinet Habu — Ramesses III Temple       Luxor      200.0                2.00                82 0.7467        [historical, pharaonic, temple, rames

# Step 4 — K-Means Clustering (Geographic Day Grouping)

In [65]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# --- 4.1 Prepare Geographic Features ---
coords = df_scored[["latitude", "longitude"]].values

# Scale the coordinates so K-Means works properly
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

n_days = USER_INPUT["duration_days"]

# --- 4.2 Run K-Means ---
kmeans = KMeans(
    n_clusters=n_days,
    init="k-means++",   # Better than random initialization
    n_init=20,          # Try 20 times and choose the best result
    random_state=42
)

df_scored["day_cluster"] = kmeans.fit_predict(coords_scaled)

# --- 4.3 Label Clusters as Day 1, Day 2, ... ---
# Sort clusters from north to south
# (higher latitude = earlier day)
cluster_centers = scaler.inverse_transform(
    kmeans.cluster_centers_
)

cluster_lat_order = (
    cluster_centers[:, 0]
    .argsort()[::-1]   # descending latitude
)

cluster_to_day = {
    cluster_id: day_num + 1
    for day_num, cluster_id in enumerate(cluster_lat_order)
}

df_scored["day"] = df_scored["day_cluster"].map(cluster_to_day)

# --- 4.4 Within Each Day: Sort by Score ---
df_scored = df_scored.sort_values(
    ["day", "score"],
    ascending=[True, False]
).reset_index(drop=True)

# --- 4.5 Print Results ---
print(
    f"📅 Trip Plan — {n_days} Days in "
    f"{USER_INPUT['governorate']}"
)

print("=" * 65)

for day in range(1, n_days + 1):

    day_places = df_scored[
        df_scored["day"] == day
    ]

    day_cost = day_places["price_egp"].sum()
    day_hours = day_places["avg_duration_hours"].sum()

    print(
        f"\n🗓️  Day {day}  |  "
        f"Cost: {day_cost:.0f} EGP  |  "
        f"Hours: {day_hours:.1f}h"
    )

    print("-" * 65)

    for _, row in day_places.iterrows():

        print(f"  📍 {row['name_en']}")

        print(
            f"     💰 {row['price_egp']:.0f} EGP  |  "
            f"⏱️  {row['avg_duration_hours']}h  |  "
            f"⭐ {row['rating']}  |  "
            f"Score: {row['score']}"
        )

print("\n" + "=" * 65)

print(
    f"💰 Total estimated cost : "
    f"{df_scored['price_egp'].sum():.0f} EGP"
)

print(
    f"⏱️  Total hours          : "
    f"{df_scored['avg_duration_hours'].sum():.1f}h"
)

print(
    f"📍 Total places         : "
    f"{len(df_scored)}"
)

📅 Trip Plan — 3 Days in Luxor

🗓️  Day 1  |  Cost: 200 EGP  |  Hours: 2.5h
-----------------------------------------------------------------
  📍 Temple of Seti I — Abydos
     💰 200 EGP  |  ⏱️  2.5h  |  ⭐ 4.8  |  Score: 0.7492

🗓️  Day 2  |  Cost: 200 EGP  |  Hours: 2.5h
-----------------------------------------------------------------
  📍 Dendera Temple — Hathor
     💰 200 EGP  |  ⏱️  2.5h  |  ⭐ 4.8  |  Score: 0.7542

🗓️  Day 3  |  Cost: 1750 EGP  |  Hours: 16.6h
-----------------------------------------------------------------
  📍 Mummification Museum Luxor
     💰 150 EGP  |  ⏱️  1.5h  |  ⭐ 4.5  |  Score: 0.75
  📍 Medinet Habu — Ramesses III Temple
     💰 200 EGP  |  ⏱️  2.0h  |  ⭐ 4.7  |  Score: 0.7467
  📍 Avenue of Sphinxes
     💰 0 EGP  |  ⏱️  1.5h  |  ⭐ 4.8  |  Score: 0.73
  📍 Ramesseum — Mortuary Temple of Ramesses II
     💰 200 EGP  |  ⏱️  1.5h  |  ⭐ 4.5  |  Score: 0.7267
  📍 Colossi of Memnon
     💰 0 EGP  |  ⏱️  0.62h  |  ⭐ 4.6  |  Score: 0.72
  📍 Temple of Seti I — Qurna
   

# Step 5 — Constraint Optimization & JSON Output


In [66]:
# --- 5.1 Constraints ---
MAX_HOURS_PER_DAY  = 8.0          # Maximum hours per day
MAX_BUDGET_PER_DAY = daily_budget # 1000 EGP in this example

# --- 5.2 Re-distribute Places Across Days Respecting Constraints ---
def optimize_itinerary(df_candidates, n_days, max_hours, max_budget):
    """
    Instead of relying only on clustering,
    redistribute places across days using a greedy approach:

      - Higher-score places are assigned first
      - Each place is added to the first day with enough capacity
      - If a day exceeds the constraints (hours or budget),
        try another day
    """

    # Sort places:
    # 1) By geographic cluster
    # 2) By score inside each cluster
    df_sorted = df_candidates.sort_values(
        ["day_cluster", "score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # Initialize trip days
    days = {
        i: {
            "places": [],
            "hours": 0.0,
            "cost": 0.0
        }
        for i in range(1, n_days + 1)
    }

    for _, place in df_sorted.iterrows():

        placed = False

        # Preferred day from clustering
        preferred_day = place["day"]

        # Try preferred day first,
        # then try remaining days
        day_order = (
            [preferred_day] +
            [
                d for d in range(1, n_days + 1)
                if d != preferred_day
            ]
        )

        for day in day_order:

            new_hours = (
                days[day]["hours"] +
                place["avg_duration_hours"]
            )

            new_cost = (
                days[day]["cost"] +
                place["price_egp"]
            )

            # Check constraints
            if (
                new_hours <= max_hours and
                new_cost  <= max_budget
            ):

                days[day]["places"].append(place)

                days[day]["hours"] += (
                    place["avg_duration_hours"]
                )

                days[day]["cost"] += (
                    place["price_egp"]
                )

                placed = True
                break

        # If not placed, skip this place
        if not placed:
            pass

    return days


optimized = optimize_itinerary(
    df_scored,
    n_days,
    MAX_HOURS_PER_DAY,
    MAX_BUDGET_PER_DAY
)

# --- 5.3 Print Optimized Plan ---
print(
    f"📅 Optimized Trip Plan — "
    f"{n_days} Days in "
    f"{USER_INPUT['governorate']}"
)

print("=" * 65)

total_cost   = 0
total_hours  = 0
total_places = 0

for day, data in optimized.items():

    print(
        f"\n🗓️  Day {day}  |  "
        f"Cost: {data['cost']:.0f} EGP  |  "
        f"Hours: {data['hours']:.1f}h  |  "
        f"Places: {len(data['places'])}"
    )

    print("-" * 65)

    for p in data["places"]:

        print(f"  📍 {p['name_en']}")

        print(
            f"     💰 {p['price_egp']:.0f} EGP  |  "
            f"⏱️  {p['avg_duration_hours']}h  |  "
            f"⭐ {p['rating']}"
        )

    total_cost   += data["cost"]
    total_hours  += data["hours"]
    total_places += len(data["places"])

print("\n" + "=" * 65)

print(
    f"💰 Total cost    : "
    f"{total_cost:.0f} / "
    f"{USER_INPUT['budget_egp']} EGP  "
    f"({total_cost / USER_INPUT['budget_egp'] * 100:.1f}%)"
)

print(
    f"⏱️  Total hours   : "
    f"{total_hours:.1f}h"
)

print(
    f"📍 Total places  : "
    f"{total_places}"
)

📅 Optimized Trip Plan — 3 Days in Luxor

🗓️  Day 1  |  Cost: 800 EGP  |  Hours: 7.0h  |  Places: 4
-----------------------------------------------------------------
  📍 Temple of Seti I — Qurna
     💰 200 EGP  |  ⏱️  1.5h  |  ⭐ 4.5
  📍 Hatshepsut Temple — Deir el-Bahari
     💰 350 EGP  |  ⏱️  2.5h  |  ⭐ 4.8
  📍 Medamud Temple
     💰 100 EGP  |  ⏱️  1.5h  |  ⭐ 4.2
  📍 Temple of Amenhotep III — Luxor West
     💰 150 EGP  |  ⏱️  1.5h  |  ⭐ 4.3

🗓️  Day 2  |  Cost: 800 EGP  |  Hours: 7.5h  |  Places: 3
-----------------------------------------------------------------
  📍 Luxor Temple
     💰 400 EGP  |  ⏱️  2.5h  |  ⭐ 4.8
  📍 Temple of Seti I — Abydos
     💰 200 EGP  |  ⏱️  2.5h  |  ⭐ 4.8
  📍 Dendera Temple — Hathor
     💰 200 EGP  |  ⏱️  2.5h  |  ⭐ 4.8

🗓️  Day 3  |  Cost: 550 EGP  |  Hours: 7.1h  |  Places: 5
-----------------------------------------------------------------
  📍 Mummification Museum Luxor
     💰 150 EGP  |  ⏱️  1.5h  |  ⭐ 4.5
  📍 Medinet Habu — Ramesses III Temple
     💰 2

In [67]:
# --- 5.4 Build JSON Output ---
itinerary_json = {
    "trip_summary": {
        "governorate"      : USER_INPUT["governorate"],
        "duration_days"    : USER_INPUT["duration_days"],
        "total_budget_egp" : USER_INPUT["budget_egp"],
        "total_cost_egp"   : round(total_cost, 2),

        "budget_used_pct"  : round(
            total_cost /
            USER_INPUT["budget_egp"] * 100,
            1
        ),

        "total_places"     : total_places,
        "total_hours"      : round(total_hours, 1),
        "interests"        : USER_INPUT["interests"],
    },

    "itinerary": []
}

for day, data in optimized.items():

    day_entry = {
        "day"            : day,
        "total_cost_egp" : round(data["cost"], 2),
        "total_hours"    : round(data["hours"], 1),
        "places"         : []
    }

    for p in data["places"]:

        day_entry["places"].append({

            "id"                 : int(p["id"]),
            "name_ar"            : p["name_ar"],
            "name_en"            : p["name_en"],
            "category"           : p["category"],

            "price_egp"          : float(p["price_egp"]),

            "avg_duration_hours" : float(
                p["avg_duration_hours"]
            ),

            "rating"             : float(p["rating"]),
            "score"              : float(p["score"]),

            "latitude"           : float(p["latitude"]),
            "longitude"          : float(p["longitude"]),

            "tags"               : p["tags_clean"],

            "opening_hours"      : p["opening_hours"],

            "image_url"          : p["image_url"],
        })

    itinerary_json["itinerary"].append(day_entry)

# --- 5.5 Save JSON ---
output_filename = (
    f"itinerary_"
    f"{USER_INPUT['governorate'].lower()}_"
    f"{n_days}days.json"
)

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(
        itinerary_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"\n💾 JSON saved: {output_filename}")

print("\n📄 JSON Preview (trip_summary):")

print(
    json.dumps(
        itinerary_json["trip_summary"],
        ensure_ascii=False,
        indent=2
    )
)


💾 JSON saved: itinerary_luxor_3days.json

📄 JSON Preview (trip_summary):
{
  "governorate": "Luxor",
  "duration_days": 3,
  "total_budget_egp": 3000,
  "total_cost_egp": 2150.0,
  "budget_used_pct": 71.7,
  "total_places": 12,
  "total_hours": 21.6,
  "interests": [
    "historical",
    "pharaonic",
    "temple",
    "museum"
  ]
}


# Step 6 — Validation & Quality Report


In [68]:
# --- 6.1 Validation Rules ---
RULES = {
    "max_hours_per_day"   : MAX_HOURS_PER_DAY,
    "max_budget_per_day"  : MAX_BUDGET_PER_DAY,
    "min_places_per_day"  : 1,
    "max_places_per_day"  : 6,

    # If the model recommends only free places
    "min_budget_used_pct" : 30.0,

    # Recommendation quality threshold
    "min_avg_score"       : 0.50,
}

# --- 6.2 Run Validation ---
errors   = []
warnings = []
passed   = []

summary = itinerary_json["trip_summary"]
days    = itinerary_json["itinerary"]

In [69]:
# ① JSON Structure Check
required_summary_fields = [
    "governorate",
    "duration_days",
    "total_budget_egp",
    "total_cost_egp",
    "budget_used_pct",
    "total_places",
    "total_hours",
    "interests"
]

required_place_fields = [
    "id",
    "name_ar",
    "name_en",
    "category",
    "price_egp",
    "avg_duration_hours",
    "rating",
    "score",
    "latitude",
    "longitude",
    "tags",
    "opening_hours",
    "image_url"
]

missing_summary = [
    f for f in required_summary_fields
    if f not in summary
]

if missing_summary:

    errors.append(
        f"❌ Missing summary fields: "
        f"{missing_summary}"
    )

else:

    passed.append(
        "✅ JSON structure — "
        "all required fields present"
    )

for day_data in days:

    for place in day_data["places"]:

        missing_place = [
            f for f in required_place_fields
            if f not in place
        ]

        if missing_place:

            errors.append(
                f"❌ Place "
                f"'{place.get('name_en', '?')}' "
                f"missing fields: {missing_place}"
            )

if not any("Missing fields" in e for e in errors):

    passed.append(
        "✅ Place fields — "
        "all required fields present "
        "in every place"
    )

In [70]:
# ② Per-Day Constraints
for day_data in days:

    d = day_data["day"]
    h = day_data["total_hours"]
    c = day_data["total_cost_egp"]
    n = len(day_data["places"])

    # Hours validation
    if h > RULES["max_hours_per_day"]:

        errors.append(
            f"❌ Day {d}: "
            f"{h}h exceeds max "
            f"{RULES['max_hours_per_day']}h"
        )

    else:

        passed.append(
            f"✅ Day {d}: hours OK "
            f"({h}h ≤ "
            f"{RULES['max_hours_per_day']}h)"
        )

    # Budget validation
    if c > RULES["max_budget_per_day"]:

        errors.append(
            f"❌ Day {d}: "
            f"{c} EGP exceeds "
            f"daily budget "
            f"{RULES['max_budget_per_day']} EGP"
        )

    else:

        passed.append(
            f"✅ Day {d}: budget OK "
            f"({c:.0f} EGP ≤ "
            f"{RULES['max_budget_per_day']:.0f} EGP)"
        )

    # Number of places validation
    if n < RULES["min_places_per_day"]:

        errors.append(
            f"❌ Day {d}: only "
            f"{n} place(s) — too few"
        )

    elif n > RULES["max_places_per_day"]:

        warnings.append(
            f"⚠️  Day {d}: "
            f"{n} places — "
            f"consider reducing for comfort"
        )

    else:

        passed.append(
            f"✅ Day {d}: place count OK "
            f"({n} places)"
        )

In [71]:
# ③ Budget Utilization

budget_pct = summary["budget_used_pct"]

if budget_pct < RULES["min_budget_used_pct"]:

    warnings.append(
        f"⚠️  Budget utilization low: "
        f"{budget_pct}% — "
        f"model may be too conservative"
    )

elif budget_pct > 100:

    errors.append(
        f"❌ Budget exceeded: "
        f"{budget_pct}%"
    )

else:

    passed.append(
        f"✅ Budget utilization: "
        f"{budget_pct}% "
        f"(within range)"
    )

In [72]:
# ④ Score Quality
all_scores = [
    p["score"]
    for d in days
    for p in d["places"]
]

avg_score = (
    sum(all_scores) / len(all_scores)
    if all_scores else 0
)

if avg_score < RULES["min_avg_score"]:

    warnings.append(
        f"⚠️  Average score low: "
        f"{avg_score:.3f} — "
        f"recommendations may not match "
        f"interests well"
    )

else:

    passed.append(
        f"✅ Recommendation quality: "
        f"avg score {avg_score:.3f} "
        f"(≥ {RULES['min_avg_score']})"
    )

In [73]:
# ⑤ Coordinate Sanity Check
for day_data in days:

    for place in day_data["places"]:

        lat = place["latitude"]
        lng = place["longitude"]

        if not (
            22.0 <= lat <= 32.0 and
            25.0 <= lng <= 37.0
        ):

            errors.append(
                f"❌ '{place['name_en']}': "
                f"coordinates out of Egypt bounds "
                f"({lat}, {lng})"
            )

if not any("coordinates" in e for e in errors):

    passed.append(
        "✅ Coordinates — "
        "all places within Egypt bounds"
    )

In [74]:
# ⑥ Duplicate Places Check
all_ids = [
    p["id"]
    for d in days
    for p in d["places"]
]

if len(all_ids) != len(set(all_ids)):

    errors.append(
        "❌ Duplicate places detected "
        "in itinerary"
    )

else:

    passed.append(
        "✅ No duplicate places"
    )


In [75]:
# 6.3 Print Validation Report
print("=" * 65)
print("           🔍 VALIDATION & QUALITY REPORT")
print("=" * 65)

print(f"\n✅ PASSED ({len(passed)})")

for p in passed:
    print(f"   {p}")

if warnings:

    print(f"\n⚠️  WARNINGS ({len(warnings)})")

    for w in warnings:
        print(f"   {w}")

if errors:

    print(f"\n❌ ERRORS ({len(errors)})")

    for e in errors:
        print(f"   {e}")

else:

    print(f"\n🎉 NO ERRORS FOUND")



# 6.4 Final Summary Card
status = (
    "🟢 PASSED"
    if not errors
    else "🔴 FAILED"
)

print("\n" + "=" * 65)

print(f"  VALIDATION STATUS  :  {status}")

print(
    f"  Governorate        :  "
    f"{summary['governorate']}"
)

print(
    f"  Duration           :  "
    f"{summary['duration_days']} days"
)

print(
    f"  Total Places       :  "
    f"{summary['total_places']}"
)

print(
    f"  Total Cost         :  "
    f"{summary['total_cost_egp']:.0f} / "
    f"{summary['total_budget_egp']} EGP"
)

print(
    f"  Budget Used        :  "
    f"{summary['budget_used_pct']}%"
)

print(
    f"  Total Hours        :  "
    f"{summary['total_hours']}h"
)

print(
    f"  Avg Rec. Score     :  "
    f"{avg_score:.3f}"
)

print(
    f"  Checks Passed      :  "
    f"{len(passed)}"
)

print(
    f"  Warnings           :  "
    f"{len(warnings)}"
)

print(
    f"  Errors             :  "
    f"{len(errors)}"
)

print("=" * 65)

print(
    "Model pipeline finished successfully"
)

           🔍 VALIDATION & QUALITY REPORT

✅ PASSED (15)
   ✅ JSON structure — all required fields present
   ✅ Place fields — all required fields present in every place
   ✅ Day 1: hours OK (7.0h ≤ 8.0h)
   ✅ Day 1: budget OK (800 EGP ≤ 1000 EGP)
   ✅ Day 1: place count OK (4 places)
   ✅ Day 2: hours OK (7.5h ≤ 8.0h)
   ✅ Day 2: budget OK (800 EGP ≤ 1000 EGP)
   ✅ Day 2: place count OK (3 places)
   ✅ Day 3: hours OK (7.1h ≤ 8.0h)
   ✅ Day 3: budget OK (550 EGP ≤ 1000 EGP)
   ✅ Day 3: place count OK (5 places)
   ✅ Budget utilization: 71.7% (within range)
   ✅ Recommendation quality: avg score 0.727 (≥ 0.5)
   ✅ Coordinates — all places within Egypt bounds
   ✅ No duplicate places

🎉 NO ERRORS FOUND

  VALIDATION STATUS  :  🟢 PASSED
  Governorate        :  Luxor
  Duration           :  3 days
  Total Places       :  12
  Total Cost         :  2150 / 3000 EGP
  Budget Used        :  71.7%
  Total Hours        :  21.6h
  Avg Rec. Score     :  0.727
  Checks Passed      :  15
  Warnings 

# Step 7 — Save Model Artifacts


In [76]:
import pickle
from google.colab import files

# --- 7.1 Save KMeans + Scaler ---
model_artifacts = {
    "kmeans"  : kmeans,
    "scaler"  : scaler,
    "n_days"  : n_days,
    "trained_on_governorate": USER_INPUT["governorate"],
}

with open("kmeans_model.pkl", "wb") as f:
    pickle.dump(model_artifacts, f)

print("✅ Saved: kmeans_model.pkl")

# --- 7.2 Save Pipeline Config ---
pipeline_config = {
    "scoring_weights": {
        "tag_match"  : 0.50,
        "budget_fit" : 0.25,
        "popularity" : 0.25,
    },
    "constraints": {
        "max_hours_per_day"   : MAX_HOURS_PER_DAY,
        "budget_cap_ratio"    : 0.60,
        "max_places_per_trip" : "duration_days * 4",
    },
    "validation_rules": {
        "max_hours_per_day"   : MAX_HOURS_PER_DAY,
        "max_budget_per_day"  : MAX_BUDGET_PER_DAY,
        "min_places_per_day"  : 1,
        "max_places_per_day"  : 6,
        "min_budget_used_pct" : 30.0,
        "min_avg_score"       : 0.50,
    },
    "supported_governorates": ["Cairo", "Giza", "Luxor", "Aswan"],
    "data_file"             : "places_all_v1.json",
}

with open("pipeline_config.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_config, f, ensure_ascii=False, indent=2)

print("✅ Saved: pipeline_config.json")

# --- 7.3 Verify pkl loads correctly ---
with open("kmeans_model.pkl", "rb") as f:
    loaded = pickle.load(f)

print("\n🔍 Verification:")
print(f"   KMeans clusters : {loaded['kmeans'].n_clusters}")
print(f"   Scaler mean     : {loaded['scaler'].mean_.round(4)}")
print(f"   Trained on      : {loaded['trained_on_governorate']}")
print(f"   n_days          : {loaded['n_days']}")

# --- 7.4 Download both files ---
print("\n📥 Downloading files...")
files.download("kmeans_model.pkl")
files.download("pipeline_config.json")

print("\n✅ Model artifacts saved & downloaded")

✅ Saved: kmeans_model.pkl
✅ Saved: pipeline_config.json

🔍 Verification:
   KMeans clusters : 3
   Scaler mean     : [25.7952 32.5691]
   Trained on      : Luxor
   n_days          : 3

📥 Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Model artifacts saved & downloaded
